In [1]:
import os
os.chdir('../../../..')

In [5]:
import numpy as np
import polars as pl

from scipy.spatial.distance import squareform
import plotly.graph_objects as go
from matplotlib import pyplot as plt
from sklearn.preprocessing import StandardScaler
# import kmeans
from sklearn.cluster import KMeans
from scipy.cluster.hierarchy import linkage as scipy_linkage, dendrogram
from scipy.cluster.hierarchy import cophenet, fcluster
from scipy.sparse.csgraph import laplacian
from scipy.linalg import eigh
from sklearn.metrics import silhouette_score, davies_bouldin_score
from scipy.spatial.distance import pdist, squareform

from src.datasets import MaterialsProject

from scripts.materials_project.evaluation_pipeline import (run_evaluation, 
                                                           hierachial_clustering, 
                                                           get_overall_chemical_coherence, 
                                                           get_distance_matrices,
                                                           build_invariant_matrix
                                                           )
from src.helper_functions import create_chemiscope_viewer

In [3]:
mp = MaterialsProject(add_soap=True, add_acsf=False, sampling_strategy="stratified", stratify_on=["band_gap", "energy_above_hull"])
df = mp.load(limit=1000)

2026-03-31 20:22:07.139 | INFO     | src.datasets:load:1055 - Loading full cached Parquet data from data/Materials Project/materials.parquet...
2026-03-31 20:22:08.062 | INFO     | src.datasets:load:1085 - Sampling 1000 rows using stratified strategy...
2026-03-31 20:22:08.182 | INFO     | src.datasets:load:1102 - Computing descriptors on sampled subset (1000 rows) and saving to tagged cache: sample_n1000_seed40_stratified
2026-03-31 20:22:08.183 | INFO     | src.datasets:_add_descriptors:1346 - Ignoring output_tag=sample_n1000_seed40_stratified since descriptors are not saved to disk.
2026-03-31 20:22:08.183 | INFO     | src.datasets:_add_descriptors:1349 - Extracting unique elements from formulas...
2026-03-31 20:22:24.835 | INFO     | src.datasets:_add_descriptors:1359 - Found 85 unique elements. Warning: Feature vectors will be massive.
2026-03-31 20:22:24.844 | INFO     | src.datasets:_add_descriptors:1390 - Computing SOAP chunk 0 (0 to 1000)...
2026-03-31 20:22:27.556 | SUCCESS  

In [9]:
feature_keys = ['coord', 'avg_neighbor_dist', 'max_neighbor_dist']
raw_matrix = build_invariant_matrix(df, aggregated=True, feature_keys=feature_keys)
scaler = StandardScaler()
scaled_matrix = scaler.fit_transform(raw_matrix)
labels = KMeans(n_clusters=2, random_state=42).fit_predict(scaled_matrix)
sil = silhouette_score(scaled_matrix, labels)
db = davies_bouldin_score(scaled_matrix, labels)
print(f"Silhouette Score: {sil:.4f}")
print(f"Davies-Bouldin Score: {db:.4f}")

Silhouette Score: 0.5407
Davies-Bouldin Score: 0.9659


In [10]:
create_chemiscope_viewer(df,  squareform(pdist(scaled_matrix, metric='euclidean')), labels=labels, reduction_method='UMAP')

Running UMAP dimensionality reduction...


/Users/karlfindhansen/thesis/Anomaly-Detection-in-Molecular-and-Materials-Datasets/.venv/lib/python3.12/site-packages/umap/umap_.py:1865: UserWarning: using precomputed metric; inverse_transform will be unavailable
  warn("using precomputed metric; inverse_transform will be unavailable")
/Users/karlfindhansen/thesis/Anomaly-Detection-in-Molecular-and-Materials-Datasets/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Converting Pymatgen structures to ASE Atoms for Chemiscope...
Assembling properties for Chemiscope...
Generating Chemiscope widget...
Saved Chemiscope input to: materials_UMAP_clustering.json
If the viewer does not open automatically, run `chemiscope show materials_UMAP_clustering.json`.


<ChemiscopeWidget(meta={'name': 'Materials Project - SOAP UMAP Clustering'}, settings={'map': {'x': {'property…